# APSP: Algoritmo Base vs Floyd-Warshall — Experimentos

Este notebook clona el repositorio, compila el codigo C++ (algoritmos, generadores de instancias y arnes de mediciones), corre los 4 experimentos del diseno experimental y grafica los resultados en escala lineal junto a sus comportamientos teoricos ajustados.

Pensado para correr en **Google Colab** (Ubuntu + g++ + make preinstalados, no requiere nada adicional).

## 1. Clonar el repositorio

In [ ]:
REPO_URL = "https://github.com/TomasCardenasO/tarea2AnalisisDeAlgoritmos2026.git"
REPO_DIR = "tarea2AnalisisDeAlgoritmos2026"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL"
else:
    !cd "$REPO_DIR" && git pull
%cd $REPO_DIR

## 2. Compilar

`make all` compila los generadores de instancias, el arnes de experimentos y los demos de cada algoritmo.

In [ ]:
!g++ --version | head -1
!make --version | head -1
!make all

## 3. Generar las instancias de los 4 experimentos

Deterministico (seeds fijas): siempre genera exactamente las mismas instancias. Ya vienen versionadas en `instances/`, pero `make instances` las regenera desde cero si hace falta (por ejemplo, si se cambian los generadores).

In [ ]:
!make instances

## 4. Calibracion rapida (opcional pero recomendado)

Antes de lanzar la corrida real con `RUNS=32` (que puede tardar bastante, sobre todo el Experimento 1 en n=200 denso y el Experimento 2 en n=1000), conviene correr una vez con pocas repeticiones para estimar cuanto va a demorar la corrida completa y confirmar que todo funciona correctamente.

In [ ]:
!make run-exp1 RUNS=4
!make run-exp2 RUNS=4
!make run-exp3 RUNS=4
!make run-exp4 RUNS=4
!echo '--- listo, revisa los tiempos de arriba para estimar la corrida con RUNS=32 ---'

## 5. Corrida real (>= 32 repeticiones, exigido por el enunciado)

Cada celda es independiente; se pueden correr por separado si se quiere ir revisando resultados a medida que van terminando. Los resultados quedan en `results/resultados_exp{1,2,3,4}.csv`.

In [ ]:
!make run-exp1 RUNS=32

In [ ]:
!make run-exp2 RUNS=32

In [ ]:
!make run-exp3 RUNS=32

In [ ]:
!make run-exp4 RUNS=32

## 6. Graficar resultados

Escala lineal en ambos ejes, comparando ambos algoritmos por experimento, con barras de error (desviacion estandar) y la curva teorica ajustada por minimos cuadrados sobre los datos medidos (mismo exponente que predice el analisis asintotico, constante ajustada empiricamente).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def cargar(exp):
    return pd.read_csv(f"results/resultados_{exp}.csv")

def serie(df, algoritmo):
    d = df[df["algoritmo"] == algoritmo].copy()
    return d

def ajustar_potencia(x, t, k):
    """Ajusta t = a * x^k (minimos cuadrados, una sola constante) y retorna a."""
    x = np.asarray(x, dtype=float); t = np.asarray(t, dtype=float)
    xk = x ** k
    a = np.sum(t * xk) / np.sum(xk * xk)
    return a

def ajustar_lineal(x, t):
    """Ajusta t = a*x + b por minimos cuadrados."""
    a, b = np.polyfit(x, t, 1)
    return a, b

### Experimento 1 — grafos densos (m ~ n^2): O(n^4) vs O(n^3)

In [ ]:
df1 = cargar("exp1")
base1 = serie(df1, "AlgoritmoBase")
fw1 = serie(df1, "FloydWarshall")

a_base1 = ajustar_potencia(base1["n"], base1["t_mean_ns"], 4)
a_fw1 = ajustar_potencia(fw1["n"], fw1["t_mean_ns"], 3)
n_teo = np.linspace(base1["n"].min(), base1["n"].max(), 200)

plt.figure(figsize=(7, 5))
plt.errorbar(base1["n"], base1["t_mean_ns"], yerr=base1["t_stdev_ns"], fmt="o", label="Algoritmo Base (medido)", color="tab:red")
plt.errorbar(fw1["n"], fw1["t_mean_ns"], yerr=fw1["t_stdev_ns"], fmt="o", label="Floyd-Warshall (medido)", color="tab:blue")
plt.plot(n_teo, a_base1 * n_teo ** 4, "--", color="tab:red", label=f"ajuste O(n^4): a={a_base1:.3g}")
plt.plot(n_teo, a_fw1 * n_teo ** 3, "--", color="tab:blue", label=f"ajuste O(n^3): a={a_fw1:.3g}")
plt.xlabel("n (numero de vertices)")
plt.ylabel("tiempo promedio (ns)")
plt.title("Experimento 1: grafos densos (m ~ n^2)")
plt.legend()
plt.tight_layout()
plt.savefig("results/exp1.png", dpi=150)
plt.show()

### Experimento 2 — grafos dispersos (m = O(n)): ambos O(n^3)

In [ ]:
df2 = cargar("exp2")
base2 = serie(df2, "AlgoritmoBase")
fw2 = serie(df2, "FloydWarshall")

a_base2 = ajustar_potencia(base2["n"], base2["t_mean_ns"], 3)
a_fw2 = ajustar_potencia(fw2["n"], fw2["t_mean_ns"], 3)
n_teo = np.linspace(base2["n"].min(), base2["n"].max(), 200)

plt.figure(figsize=(7, 5))
plt.errorbar(base2["n"], base2["t_mean_ns"], yerr=base2["t_stdev_ns"], fmt="o", label="Algoritmo Base (medido)", color="tab:red")
plt.errorbar(fw2["n"], fw2["t_mean_ns"], yerr=fw2["t_stdev_ns"], fmt="o", label="Floyd-Warshall (medido)", color="tab:blue")
plt.plot(n_teo, a_base2 * n_teo ** 3, "--", color="tab:red", label=f"ajuste O(n^3): a={a_base2:.3g}")
plt.plot(n_teo, a_fw2 * n_teo ** 3, "--", color="tab:blue", label=f"ajuste O(n^3): a={a_fw2:.3g}")
plt.xlabel("n (numero de vertices)")
plt.ylabel("tiempo promedio (ns)")
plt.title("Experimento 2: grafos dispersos (m = O(n))")
plt.legend()
plt.tight_layout()
plt.savefig("results/exp2.png", dpi=150)
plt.show()

### Experimento 3 — n fijo, m variable: Floyd-Warshall constante vs Base lineal en m

In [ ]:
df3 = cargar("exp3")
base3 = serie(df3, "AlgoritmoBase")
fw3 = serie(df3, "FloydWarshall")

a_base3, b_base3 = ajustar_lineal(base3["m"], base3["t_mean_ns"])
c_fw3 = fw3["t_mean_ns"].mean()  # Floyd-Warshall: teoricamente independiente de m
m_teo = np.linspace(base3["m"].min(), base3["m"].max(), 200)

plt.figure(figsize=(7, 5))
plt.errorbar(base3["m"], base3["t_mean_ns"], yerr=base3["t_stdev_ns"], fmt="o", label="Algoritmo Base (medido)", color="tab:red")
plt.errorbar(fw3["m"], fw3["t_mean_ns"], yerr=fw3["t_stdev_ns"], fmt="o", label="Floyd-Warshall (medido)", color="tab:blue")
plt.plot(m_teo, a_base3 * m_teo + b_base3, "--", color="tab:red", label=f"ajuste O(m): a={a_base3:.3g}")
plt.axhline(c_fw3, linestyle="--", color="tab:blue", label=f"ajuste O(1) en m: c={c_fw3:.3g}")
plt.xlabel("m (numero de aristas), n=150 fijo")
plt.ylabel("tiempo promedio (ns)")
plt.title("Experimento 3: n fijo, densidad variable")
plt.legend()
plt.tight_layout()
plt.savefig("results/exp3.png", dpi=150)
plt.show()

### Experimento 4 — sensibilidad a la topologia / orden de procesamiento

No es una curva en funcion de n o m (n y m quedan fijos): se comparan 4 barras (2 topologias x 2 algoritmos).

In [ ]:
df4 = cargar("exp4")

casos = ["mejor", "peor"]
algoritmos = ["FloydWarshall", "AlgoritmoBase"]
x = np.arange(len(casos))
ancho = 0.35

plt.figure(figsize=(7, 5))
for i, alg in enumerate(algoritmos):
    medias = [df4[(df4["caso"] == c) & (df4["algoritmo"] == alg)]["t_mean_ns"].iloc[0] for c in casos]
    errores = [df4[(df4["caso"] == c) & (df4["algoritmo"] == alg)]["t_stdev_ns"].iloc[0] for c in casos]
    color = "tab:blue" if alg == "FloydWarshall" else "tab:red"
    plt.bar(x + (i - 0.5) * ancho, medias, ancho, yerr=errores, label=alg, color=color)

plt.xticks(x, ["mejor caso\n(orden ascendente)", "peor caso\n(orden descendente)"])
plt.ylabel("tiempo promedio (ns)")
plt.title("Experimento 4: sensibilidad a la topologia/orden (n=1000, m=999)")
plt.legend()
plt.tight_layout()
plt.savefig("results/exp4.png", dpi=150)
plt.show()

## 7. (Opcional) Descargar resultados y graficos

Para llevarse los CSV y PNG generados de vuelta a la maquina local.

In [ ]:
import shutil
shutil.make_archive("resultados", "zip", "results")

try:
    from google.colab import files
    files.download("resultados.zip")
except ImportError:
    print("No se detecto entorno Colab; el archivo resultados.zip quedo en el directorio actual.")